# Train the construction-safety-monitor models on a free Colab GPU

This notebook fine-tunes the models the app uses:
1. **YOLO26n PPE detector** — person + helmet/vest/gloves/boots/mask as boxes (switched from YOLOv12n: Ultralytics' own docs note YOLO12's attention-heavy blocks trade away CPU throughput and training stability for accuracy that doesn't clearly show up at the nano size - and this project hit that instability firsthand during a local training attempt)
2. **ViT PPE classifier** — the model the live app actually loads by default (`python/models/vit_ppe_classifier/`)
3. **ViT fall classifier** — backs "fallen" detection, replacing the crude aspect-ratio heuristic currently in use

**Before running:** Runtime → Change runtime type → T4 GPU (or better).

You'll be prompted to upload 4 files from your machine the first time through (built earlier by the assistant) — only the code needs re-uploading on later runs, the dataset zips get cached in Drive:
- `csm_code.zip` — the project's `python/` source (small, seconds to upload)
- `ppe_training_package.zip` — the ViT PPE dataset (16,025 images, ~570MB)
- `yolo_training_package.zip` — the merged YOLO dataset (~9,500 images, resized)
- `fall_training_package.zip` — the ViT fall dataset (~4,875 crops, merged from two public Roboflow sources)

This notebook mounts Google Drive (section 2.5) to cache the datasets and checkpoint training progress — see that section for why. If the Colab runtime disconnects partway through, don't restart from scratch: re-run the cells in order from the top and everything resumes automatically (code needs a fresh upload since it's small anyway; the big datasets and any training progress are picked up from Drive instead of re-uploaded).

In [ ]:
!nvidia-smi

## 1. Upload the code

In [ ]:
import os
os.makedirs('/content/csm/python', exist_ok=True)
%cd /content/csm/python
from google.colab import files
print('Select csm_code.zip')
uploaded = files.upload()
import zipfile
zipfile.ZipFile('csm_code.zip').extractall('.')
!ls

## 2. Install dependencies
Colab already ships a CUDA-matched torch/torchvision — don't touch those, just add what's missing.

In [ ]:
!pip install -q ultralytics "transformers>=4.40.0,<5.0.0" accelerate scikit-learn tqdm pyyaml

## 2.5 Mount Google Drive — makes disconnects cheap

Colab's free tier can disconnect on idle timeout or the session cap, and when it does, **it hands you a brand-new empty VM** — nothing on `/content` survives, including any training in progress. Mounting Drive here means:
- The two dataset zips only need to be uploaded from your laptop *once* — after that they're cached in Drive and just get copied over (fast) instead of re-uploaded.
- Both training scripts save their checkpoints straight to Drive as they go, and will auto-resume from there instead of starting over, if you have to re-run this notebook after a disconnect.

You'll get a Google auth popup — approve it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/csm_training'
DRIVE_CACHE = f'{DRIVE_DIR}/dataset_cache'
DRIVE_RUNS = f'{DRIVE_DIR}/runs'
DRIVE_VIT_OUT = f'{DRIVE_DIR}/vit_ppe_classifier'
DRIVE_FALL_OUT = f'{DRIVE_DIR}/vit_fall_classifier'
os.makedirs(DRIVE_CACHE, exist_ok=True)
os.makedirs(DRIVE_RUNS, exist_ok=True)
os.makedirs(DRIVE_VIT_OUT, exist_ok=True)
os.makedirs(DRIVE_FALL_OUT, exist_ok=True)
print('Drive mounted. Persistent training state lives under', DRIVE_DIR)

## 3. Upload the ViT (PPE) dataset
First time only — after this it's cached in Drive and later re-runs just copy it over instead of re-uploading 570MB from your laptop again.

In [ ]:
%cd /content/csm/python
import zipfile

ppe_zip_cache = f'{DRIVE_CACHE}/ppe_training_package.zip'
if os.path.exists(ppe_zip_cache):
    print('Found cached copy in Drive - copying instead of re-uploading')
    import shutil
    shutil.copy(ppe_zip_cache, 'ppe_training_package.zip')
else:
    print('Select ppe_training_package.zip')
    uploaded = files.upload()
    import shutil
    shutil.copy('ppe_training_package.zip', ppe_zip_cache)
    print('Cached to Drive for next time:', ppe_zip_cache)

os.makedirs('/content/csm_data', exist_ok=True)
zipfile.ZipFile('ppe_training_package.zip').extractall('/content/csm_data/ppe_pkg')
!rm -rf dataset_ppe
!mv /content/csm_data/ppe_pkg/dataset_ppe dataset_ppe
!echo 'train:' && wc -l < dataset_ppe/train_annotations.csv
!echo 'val:' && wc -l < dataset_ppe/val_annotations.csv
!ls dataset_ppe/images | wc -l

## 4. Upload the YOLO PPE-detection dataset
Same deal — cached in Drive after the first upload.

In [ ]:
%cd /content/csm/python

yolo_zip_cache = f'{DRIVE_CACHE}/yolo_training_package.zip'
if os.path.exists(yolo_zip_cache):
    print('Found cached copy in Drive - copying instead of re-uploading')
    import shutil
    shutil.copy(yolo_zip_cache, 'yolo_training_package.zip')
else:
    print('Select yolo_training_package.zip')
    uploaded = files.upload()
    import shutil
    shutil.copy('yolo_training_package.zip', yolo_zip_cache)
    print('Cached to Drive for next time:', yolo_zip_cache)

os.makedirs('datasets', exist_ok=True)
zipfile.ZipFile('yolo_training_package.zip').extractall('datasets/yolo_ppe')
!cat datasets/yolo_ppe/data.yaml
!ls datasets/yolo_ppe/images/train | wc -l
!ls datasets/yolo_ppe/images/val | wc -l

## 5. Train YOLO26n as a PPE box detector
On a T4 this should run at roughly 1-3 min/epoch (vs. ~66 min/epoch on the CPU this was originally tried on) - adjust `--epochs` down if you're short on Colab session time.

`--project` points at Drive, so `last.pt`/`best.pt` are saved there after every epoch. **If the runtime disconnects**, just re-run cells 1 → up through this one (re-mount Drive, re-copy the cached datasets, no re-upload needed) — this cell will detect the in-progress run in Drive and resume instead of starting over.

In [ ]:
%cd /content/csm/python
!python train_yolo.py --epochs 100 --imgsz 640 --batch 32 --name ppe_yolo --project "{DRIVE_RUNS}"

## 6. Train the ViT PPE classifier
This is the model the live app actually loads by default.

Updated since the first run: the training script now uses **per-label class-weighted loss** and **data augmentation** (random crop/color-jitter/blur/rotation) - added after a per-label evaluation of the first checkpoint showed `mask` was badly imbalanced (precision 1.0 but recall 0.35 - the model almost never predicted "mask present") and `gloves`/`helmet` were weaker on real-world-style images than the validation numbers suggested. It'll auto-resume from the existing Drive checkpoint and pick up these fixes on top of it. Each epoch now also prints a per-label F1 breakdown, and "best" is now selected on the **macro-average** F1 across labels (not one score dominated by the common labels) so an improvement on a rare label like mask actually counts.

`--output_dir` points at Drive, so the best checkpoint so far is saved there whenever it improves. **If the runtime disconnects**, re-run cells 1 → up through this one — it'll detect the existing checkpoint in Drive and resume fine-tuning from it instead of starting over from the base pretrained model.

In [ ]:
%cd /content/csm/python
!python train_vit.py --model ppe --epochs 20 --batch_size 64 --output_dir "{DRIVE_VIT_OUT}"

## 6.5 Upload the fall-detection dataset
First time only — cached in Drive after this, same as the PPE dataset. Built by merging two public Roboflow datasets (a large fall-only set + a smaller fall/not-fallen set) via `tools/build_dataset.py fall` — already done locally, this just uploads the converted result.

In [ ]:
%cd /content/csm/python
import zipfile

fall_zip_cache = f'{DRIVE_CACHE}/fall_training_package.zip'
if os.path.exists(fall_zip_cache):
    print('Found cached copy in Drive - copying instead of re-uploading')
    import shutil
    shutil.copy(fall_zip_cache, 'fall_training_package.zip')
else:
    print('Select fall_training_package.zip')
    uploaded = files.upload()
    import shutil
    shutil.copy('fall_training_package.zip', fall_zip_cache)
    print('Cached to Drive for next time:', fall_zip_cache)

os.makedirs('/content/csm_data', exist_ok=True)
zipfile.ZipFile('fall_training_package.zip').extractall('/content/csm_data/fall_pkg')
!rm -rf dataset_fall
!mv /content/csm_data/fall_pkg/dataset_fall dataset_fall
!echo 'train:' && wc -l < dataset_fall/train_annotations.csv
!echo 'val:' && wc -l < dataset_fall/val_annotations.csv
!ls dataset_fall/images | wc -l

## 6.6 Train the fall/not-fallen classifier
This model backs the "fallen" detection currently running on a crude aspect-ratio heuristic in the live app.

The merged dataset here has a real imbalance in the opposite direction from PPE's mask problem: most images (from one of the two source datasets) are fall-positive, with only a small number of "standing normally" negative examples. The same automatic per-label class-weighted loss used for the PPE classifier applies here too and corrects for it automatically (it computes the weight from whatever imbalance actually exists in the data, in either direction) - no code changes needed, just run it.

`--output_dir` points at Drive (`DRIVE_FALL_OUT`), same resume behavior as the PPE classifier if the runtime disconnects.

In [ ]:
%cd /content/csm/python
!python train_vit.py --model fall --epochs 15 --batch_size 64 --output_dir "{DRIVE_FALL_OUT}"

## 7. Package and download the trained weights

In [ ]:
%cd /content/csm/python
import glob, shutil
os.makedirs('/content/results', exist_ok=True)

# ViT PPE checkpoint - lives in Drive already (DRIVE_VIT_OUT), just zip it up
if os.path.isdir(DRIVE_VIT_OUT) and os.listdir(DRIVE_VIT_OUT):
    shutil.make_archive('/content/results/vit_ppe_classifier', 'zip', DRIVE_VIT_OUT)

# ViT fall checkpoint - lives in Drive already (DRIVE_FALL_OUT)
if os.path.isdir(DRIVE_FALL_OUT) and os.listdir(DRIVE_FALL_OUT):
    shutil.make_archive('/content/results/vit_fall_classifier', 'zip', DRIVE_FALL_OUT)

# YOLO best.pt - lives under DRIVE_RUNS already
yolo_runs = sorted(glob.glob(f'{DRIVE_RUNS}/ppe_yolo*/weights/best.pt'), key=os.path.getmtime)
if yolo_runs:
    shutil.copy(yolo_runs[-1], '/content/results/yolo_ppe_best.pt')

!ls -la /content/results
for f in os.listdir('/content/results'):
    files.download(f'/content/results/{f}')

## After downloading

- **ViT PPE**: unzip `vit_ppe_classifier.zip` into `python/models/vit_ppe_classifier/` on your machine (or wherever you deploy from), then restart the app - `ppe_classifier.py` picks it up automatically.
- **ViT fall**: unzip `vit_fall_classifier.zip` into `python/models/vit_fall_classifier/` - once present, the app switches "fallen" detection from the aspect-ratio heuristic to this trained model automatically, same mechanism as PPE.
- **YOLO**: put `yolo_ppe_best.pt` anywhere and point `config.py`'s `YOLO_WEIGHTS` (or the `YOLO_WEIGHTS` env var) at it. Its classes are `0=person, 1=helmet, 2=vest, 3=gloves, 4=boots, 5=mask` - `YOLO_PERSON_CLASS_ID` stays `0` so person tracking keeps working unchanged.
  - Note: the live pipeline still gets PPE presence from the ViT classifier by default, not from this YOLO model's boxes - wiring detector.py/inference_worker.py to use YOLO's PPE boxes instead (or as a second signal) is a separate follow-up, not done by this notebook.

All checkpoints are also already sitting permanently in your Google Drive under `MyDrive/csm_training/` (`vit_ppe_classifier/`, `vit_fall_classifier/`, and `runs/ppe_yolo*/weights/`), even if you never get to run this last cell — the browser download is just a convenience, not the only copy.